# Fraus - Fine-tune do BERTimbau para deteccao de ironia

Classificacao **binaria**: 0 nao-ironico, 1 ironico. A ordem esta em
`fraus.sinais.ironia` e vale aqui tambem.

**Por que uma cabeca separada** e nao mais uma classe do sinal de texto ou de
emocao: ironia nao e um sentimento, e uma relacao entre o que o texto DIZ e o
que ele SIGNIFICA. "Que atendimento maravilhoso, so esperei 3 horas" e
lexicamente positivo e pragmaticamente negativo *ao mesmo tempo*. Como quarta
classe de satisfacao, o modelo teria de escolher uma das duas leituras -- e a
informacao de que ha CONFLITO, que e o proprio sinal de ironia, se perderia.

Requer **GPU**. Este notebook e INDEPENDENTE do 01 e do 03.

## O gargalo deste notebook e o CORPUS, nao o codigo

O treino abaixo roda em qualquer corpus PT-BR com uma coluna de texto e uma de
rotulo binario: a celula 3 **detecta o esquema e falha nomeando as colunas que
achou**, em vez de presumir formato. O que falta e o corpus.

### Primario: IDPT 2021 -- pedir aos organizadores

A tarefa *Irony Detection in Portuguese* do IberLEF (15,2 mil tweets e 18,4 mil
noticias anotados) e a melhor opcao: anotacao revisada e citavel. **Nao tem
download aberto** -- solicite em
<https://sites.google.com/inf.ufpel.edu.br/idpt2021/>. O pedido so corre se
comecar, entao mande antes de precisar.

### Alternativas verificadas em 14/08/2026

Levantamento feito para nao repetir busca. O que **nao serve**, e por que:

| Fonte | Veredito |
|---|---|
| `arbml/multilingual_irony` (HF) | so `tweet_id` + `label`, **sem texto**; reidratar exige API paga do Twitter |
| `ramondomiingos/ptbr-irony-idioms-regionalism` (HF) | < 1.000 linhas e e *benchmark de LLM* com gabarito em prosa, nao corpus de classificacao |
| `rafaelanchieta/PiLN` (GitHub) | sao os **modelos** treinados do IDPT, nao o corpus; embeddings dependem de servidor antigo do NILC |
| IroSvA / SemEval-2018 traduzidos | **desaconselhado**: traducao automatica reintroduz o vazamento de procedencia que ja custou o primeiro fusor |

O que **pode servir**, com ressalva: um corpus de tweets rotulados por
**hashtag** (`#ironia` / `#sarcasmo`), publicado no Kaggle. A celula 2 sabe
baixa-lo. **Nao foi possivel verificar esquema nem licenca de fora do Colab**
(a pagina exige login), entao confira as duas antes de citar no relatorio.

O preco dessa fonte e metodologico e precisa ser **declarado**: o rotulo e
*auto-atribuido pelo autor do tweet*, nao anotado por terceiro. Isso e ruido
conhecido -- ha ironia sem hashtag, e hashtag usada como piada sem ironia. E
uma limitacao citavel, na mesma prateleira da traducao automatica do corpus de
emocao; nao e um defeito escondido.

### Se nada fechar

A saida honesta e **declarar a ironia como trabalho futuro** no relatorio. Os
modulos `fraus/sinais/ironia.py` e seus testes ja existem e continuam validos:
o que falta e o artefato treinado, e dizer isso vale mais que treinar em corpus
improprio para preencher uma linha da banca.

In [ ]:
# 1. GPU conferida; Drive montado se der.
!pip -q install transformers datasets accelerate evaluate scikit-learn

import os

import torch

assert torch.cuda.is_available(), (
    'Runtime sem GPU. Ambiente de execucao > Alterar tipo de ambiente de execucao > GPU.'
)
print('gpu:', torch.cuda.get_device_name(0))

# O Drive e CONVENIENCIA, nao requisito: ele guarda o checkpoint entre quedas
# de sessao. Falha de mount (popup bloqueado, cookie de terceiro, sessao
# zumbi) NAO pode matar o notebook na primeira celula e queimar a GPU que ja
# foi alocada -- entao cai para /content e avisa alto o que se perde.
DRIVE = '/content/drive/MyDrive/fraus'
try:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=True)
    RAIZ = DRIVE
    print('checkpoint no Drive: sobrevive a queda de sessao.')
except Exception as erro:
    RAIZ = '/content/fraus-local'
    print(f'AVISO: Drive indisponivel ({type(erro).__name__}: {erro}).')
    print(f'Gravando em {RAIZ} -- SEM rede de protecao: se a sessao cair, o')
    print('treino recomeca do zero. Baixe os artefatos ao terminar (celula 8).')

DIR_CORPUS = f'{RAIZ}/dados/ironia'
DIR_CHECKPOINT = f'{RAIZ}/checkpoints-ironia'
DIR_SAIDA = f'{RAIZ}/modelos/bertimbau-ironia'
DIR_MODELOS = os.path.dirname(DIR_SAIDA)

for pasta in (DIR_CORPUS, DIR_CHECKPOINT, DIR_SAIDA):
    os.makedirs(pasta, exist_ok=True)
print('corpus esperado em:', DIR_CORPUS)


## 2. O pacote `fraus`

Mesma celula dos notebooks 02 e 03: clone em `/content/fraus-src` (nunca
`/content/fraus`, que o cwd do Colab sequestraria como namespace package) e
`sys.path` ajustado a mao, porque o `.pth` do `pip install -e` so e lido no
boot do interpretador.

In [ ]:
# 2. Pacote fraus instalado a partir do repositorio.
DONO = 'joao-ped'
NOME_REPO = 'Fraus'
# O RAMO DO TREINO. Era 'feat/motor-e-dashboard', que em 24/08/2026 estava
# 141 commits atras do main e zero a frente -- absorvido e obsoleto.
#
# O ESTRAGO era silencioso, que e o pior tipo: naquele ramo
# `montar_features` ainda aceitava UM classificador e devolvia 16 features.
# O notebook rodava sem erro, treinava feliz, e cuspia um fusor de 16
# features num repositorio cujo `NOMES_FEATURES` tem 35. O treino nao
# falhava: ele acertava o alvo errado, e so aparecia na API, no boot, como
# 'X has 35 features, but StandardScaler is expecting 16'.
#
# Se um dia precisar treinar de um ramo de trabalho, troque aqui -- mas
# confira antes que o contrato de features daquele ramo e o do destino.
RAMO = 'main'
DIR_REPO = '/content/fraus-src'

import shutil
import subprocess
import sys

# Os TRES motivos de nao haver token sao distintos, e o remedio de cada um e
# outro. `except Exception: token = None` tratava os tres como o mesmo e caia
# para um clone anonimo -- que num repo privado morre la na frente com
# "could not read Username for 'https://github.com'", mensagem que nao
# menciona Secret nenhum e manda procurar no lugar errado. O caso que mais
# acontece e o pior de diagnosticar: o segredo existe, esta correto, e o
# toggle DESTE notebook esta desligado.
token = None
motivo_sem_token = None
try:
    from google.colab import userdata
except ImportError:
    motivo_sem_token = 'nao estamos no Colab -- nao ha painel de Secrets.'
else:
    try:
        token = userdata.get('TOKEN_GITHUB')
    except Exception as erro:
        # Comparacao por NOME da classe, e nao por `except userdata.X`, de
        # proposito: essas excecoes nao sao API estavel do Colab, e um
        # AttributeError aqui derrubaria a celula por causa do TRATAMENTO do
        # erro em vez do erro.
        classe = type(erro).__name__
        if classe == 'NotebookAccessError':
            motivo_sem_token = (
                'TOKEN_GITHUB existe, mas ESTE notebook nao tem acesso a ele. '
                'Abra o painel Secrets (icone de chave, barra da esquerda) e '
                'ligue o toggle da linha.'
            )
        elif classe == 'SecretNotFoundError':
            motivo_sem_token = 'nao ha TOKEN_GITHUB no painel Secrets deste Colab.'
        else:
            motivo_sem_token = f'{classe}: {erro}'

if motivo_sem_token:
    print('AVISO -- clone SEM credencial:', motivo_sem_token)

url_publica = f'https://github.com/{DONO}/{NOME_REPO}.git'
url = f'https://{token}@github.com/{DONO}/{NOME_REPO}.git' if token else url_publica

shutil.rmtree(DIR_REPO, ignore_errors=True)
clone = subprocess.run(
    ['git', 'clone', '-q', '--branch', RAMO, url, DIR_REPO],
    capture_output=True, text=True,
)
if clone.returncode != 0:
    # A mensagem do git traz a URL: com token dentro, isso imprimiria o segredo.
    erro = clone.stderr.replace(token, '***') if token else clone.stderr
    raise RuntimeError(
        f'git clone falhou (codigo {clone.returncode}): {erro.strip()}\n'
        'Se for erro de autenticacao, o repositorio e privado: crie um token '
        'em github.com/settings/tokens com escopo `repo`, salve no painel '
        'Secrets do Colab (icone de chave) com o nome TOKEN_GITHUB e ligue o '
        'acesso para este notebook. Alternativa sem token: suba o .zip do '
        'repositorio para /content e descompacte em ' + DIR_REPO + '.'
    )

if token:
    # `git clone` GRAVA a URL com o token em .git/config. O disco do Colab e
    # efemero, mas a saida do notebook e salva no arquivo: um `!git remote -v`
    # numa celula de debug imprimiria o segredo dentro do .ipynb, que e
    # exatamente o vazamento que usar o painel de Secrets existe para evitar.
    subprocess.run(
        ['git', '-C', DIR_REPO, 'remote', 'set-url', 'origin', url_publica],
        check=True,
    )

!pip install -q -e {DIR_REPO}

if DIR_REPO not in sys.path:
    sys.path.insert(0, DIR_REPO)

commit = subprocess.check_output(
    ['git', '-C', DIR_REPO, 'rev-parse', '--short', 'HEAD']
).decode().strip()
print('fraus instalado no commit', commit)

from fraus.sinais.ironia import IRONICO, NAO_IRONICO

print('classes:', {'nao_ironico': NAO_IRONICO, 'ironico': IRONICO})

## 3. De onde vem o corpus

Escolha `ORIGEM` abaixo. Todas alimentam a MESMA pasta (`DIR_CORPUS`) e a mesma
deteccao de esquema da celula seguinte -- trocar de fonte nao muda o resto.

- **`sintetico`** (padrao) -- gera o corpus com `fraus.ingest.gerador_ironia`.
  Nao depende de liberacao de ninguem e roda hoje.
- **`drive`** -- os arquivos ja estao em `DIR_CORPUS`.
- **`kaggle`** -- baixa da conta Kaggle (exige `KAGGLE_USERNAME`/`KAGGLE_KEY`
  nos Secrets do Colab). **Cuidado:** o corpus mais citado por hashtag e o de
  Goncalves et al. (BraSNAM 2015), coletado com `#sarcasm`/`#irony` -- ou seja,
  **em ingles**. Confira a lingua antes de treinar.
- **`upload`** -- seletor de arquivos do navegador.

### Por que o padrao e sintetico

Nao existe corpus de ironia PT-BR aberto, com texto, em tamanho treinavel --
levantamento em `docs/treinamento.md`. E o mesmo impasse do sinal de tempo, e a
mesma saida ja aceita no projeto: o fusor treina em conversas sinteticas porque
nenhum corpus de review tem timestamps.

O gerador foi escrito contra o vazamento que matou o primeiro fusor: os mesmos
fatos ruins aparecem na ironia E na reclamacao direta, as mesmas palavras
elogiosas aparecem dentro E fora da ironia, e numeros aparecem nas duas
classes. O que resta como sinal e a incongruencia. Ha testes travando cada uma
dessas propriedades (`tests/test_gerador_ironia.py`).

**A metrica interna sera otimista** -- ela mede o quanto o modelo aprendeu
estes padroes, nao o quanto le ironia humana. Por isso a celula 7 avalia
tambem num conjunto independente, se houver.

In [ ]:
# 3. Corpus trazido para DIR_CORPUS.
import glob
import shutil

ORIGEM = 'sintetico'  # 'sintetico' | 'drive' | 'kaggle' | 'upload'

# So vale para ORIGEM == 'sintetico'. O gerador esgota o espaco de
# combinacoes e devolve menos que o pedido em vez de repetir texto.
EXEMPLOS_SINTETICOS = 6000
SEMENTE = 42

# So e usado quando ORIGEM == 'kaggle'. Trocar de dataset e trocar esta linha:
# a deteccao de esquema da celula seguinte nao presume colunas.
DATASET_KAGGLE = 'viniciuscleves/irony-and-sarcasm-in-brazilian-portuguese'


def _tabelas_em(pasta):
    return sorted(glob.glob(f'{pasta}/**/*.csv', recursive=True) +
                  glob.glob(f'{pasta}/**/*.tsv', recursive=True))


if ORIGEM == 'sintetico':
    import csv

    from fraus.ingest.gerador_ironia import gerar_exemplos

    exemplos = gerar_exemplos(EXEMPLOS_SINTETICOS, semente=SEMENTE)
    destino = os.path.join(DIR_CORPUS, f'sintetico_s{SEMENTE}.csv')
    with open(destino, 'w', encoding='utf-8', newline='') as arquivo:
        escritor = csv.writer(arquivo)
        escritor.writerow(['texto', 'rotulo'])
        escritor.writerows(exemplos)
    print(f'{len(exemplos)} exemplos gerados em {destino}')
    if len(exemplos) < EXEMPLOS_SINTETICOS:
        print(f'(pedidos {EXEMPLOS_SINTETICOS}: o espaco de combinacoes acabou '
              'antes -- amplie as listas do gerador para crescer)')

elif ORIGEM == 'kaggle':
    from google.colab import userdata

    for chave in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
        try:
            os.environ[chave] = userdata.get(chave)
        except Exception:
            raise RuntimeError(
                f'{chave} ausente nos Secrets do Colab. Gere o token em '
                'https://www.kaggle.com/settings (Create New Token), guarde as '
                'duas chaves nos Secrets e libere o acesso para este notebook.'
            )

    import kagglehub

    baixado = kagglehub.dataset_download(DATASET_KAGGLE)
    print('baixado em:', baixado)
    # Copiado para DIR_CORPUS para que o resto do notebook nao saiba de onde veio.
    for origem in _tabelas_em(baixado):
        shutil.copy2(origem, DIR_CORPUS)

elif ORIGEM == 'upload':
    from google.colab import files

    enviados = files.upload()
    for nome in enviados:
        shutil.move(nome, os.path.join(DIR_CORPUS, nome))

elif ORIGEM != 'drive':
    raise ValueError(
        f"ORIGEM invalida: {ORIGEM!r}. "
        "Use 'sintetico', 'drive', 'kaggle' ou 'upload'."
    )

encontrados = _tabelas_em(DIR_CORPUS)
if not encontrados:
    raise FileNotFoundError(
        f'nenhum .csv ou .tsv em {DIR_CORPUS} (ORIGEM={ORIGEM!r}).\n'
        'IDPT 2021: solicite em https://sites.google.com/inf.ufpel.edu.br/idpt2021/ '
        'e suba os arquivos, ou use ORIGEM="kaggle"/"upload". '
        'Ver docs/treinamento.md, notebook 04.'
    )

print(f'{len(encontrados)} arquivo(s) em {DIR_CORPUS}:')
for caminho in encontrados:
    print(' ', os.path.basename(caminho))


## 4. Carga do corpus -- esquema CONFIGURAVEL

Este notebook nao presume o formato dos arquivos, porque a origem e
trocavel e o IDPT nunca foi visto por quem escreveu isto. A celula abaixo:

1. le todo `.csv` e `.tsv` da pasta, detectando o separador;
2. **imprime as colunas encontradas**;
3. procura a coluna de texto e a de rotulo pelos nomes mais provaveis;
4. **para com erro nomeando as colunas disponiveis** se nao achar.

Se a deteccao falhar, ajuste `COLUNA_TEXTO` e `COLUNA_ROTULO` a mao com o que
foi impresso. Falhar apontando o que existe e melhor do que adivinhar coluna e
treinar em lixo silenciosamente.

O mesmo vale para o rotulo: valor fora de `MAPA_ROTULO` levanta erro em vez de
virar 0 por omissao -- rotulo mal lido nao aparece como falha, aparece como
metrica ruim tres horas depois.

In [ ]:
# 4. Corpus lido e esquema detectado.
import glob

import pandas as pd

# Ajuste se a deteccao automatica falhar -- os nomes impressos abaixo mandam.
COLUNA_TEXTO = None    # ex.: 'text', 'texto', 'tweet'
COLUNA_ROTULO = None   # ex.: 'label', 'rotulo', 'irony', 'ironia'

CANDIDATOS_TEXTO = ['text', 'texto', 'tweet', 'sentence', 'frase', 'content', 'noticia']
CANDIDATOS_ROTULO = ['label', 'rotulo', 'irony', 'ironia', 'class', 'classe', 'target', 'y']

caminhos = _tabelas_em(DIR_CORPUS)  # a celula 3 ja garantiu que existe algum

partes = []
for caminho in caminhos:
    separador = '\t' if caminho.endswith('.tsv') else None  # None = sniff do pandas
    parte = pd.read_csv(caminho, sep=separador, engine='python')
    print(f'{os.path.basename(caminho):40s} {len(parte):6d} linhas  colunas={list(parte.columns)}')
    partes.append(parte)

bruto = pd.concat(partes, ignore_index=True)
colunas_minusculas = {c.lower(): c for c in bruto.columns}


def achar(candidatos, configurado, papel):
    if configurado:
        if configurado not in bruto.columns:
            raise KeyError(f'coluna {configurado!r} nao existe. Disponiveis: {list(bruto.columns)}')
        return configurado
    for candidato in candidatos:
        if candidato in colunas_minusculas:
            return colunas_minusculas[candidato]
    raise KeyError(
        f'nao achei a coluna de {papel} automaticamente. '
        f'Colunas disponiveis: {list(bruto.columns)}. '
        f'Defina COLUNA_TEXTO/COLUNA_ROTULO no topo desta celula.'
    )


coluna_texto = achar(CANDIDATOS_TEXTO, COLUNA_TEXTO, 'texto')
coluna_rotulo = achar(CANDIDATOS_ROTULO, COLUNA_ROTULO, 'rotulo')
print(f'\nusando texto={coluna_texto!r} rotulo={coluna_rotulo!r}')

dados = bruto[[coluna_texto, coluna_rotulo]].dropna()
dados.columns = ['texto', 'rotulo_bruto']
dados = dados[dados['texto'].astype(str).str.strip().astype(bool)]

# O rotulo pode vir como 0/1, "0"/"1", ou texto. Qualquer valor fora do
# mapeamento para com erro em vez de virar 0 por acidente.
MAPA_ROTULO = {
    '0': NAO_IRONICO, '1': IRONICO,
    'nao': NAO_IRONICO, 'não': NAO_IRONICO, 'no': NAO_IRONICO,
    'sim': IRONICO, 'yes': IRONICO,
    'nao-ironico': NAO_IRONICO, 'ironico': IRONICO,
    'non-irony': NAO_IRONICO, 'irony': IRONICO,
    'false': NAO_IRONICO, 'true': IRONICO,
}
chaves = dados['rotulo_bruto'].astype(str).str.strip().str.lower()
desconhecidos = sorted(set(chaves) - set(MAPA_ROTULO))
if desconhecidos:
    raise ValueError(
        f'rotulos nao reconhecidos: {desconhecidos}. '
        'Acrescente-os a MAPA_ROTULO em vez de deixar virar 0 silenciosamente.'
    )
dados['rotulo'] = chaves.map(MAPA_ROTULO).astype(int)

print(f'\n{len(dados)} exemplos')
print(dados['rotulo'].value_counts().rename({NAO_IRONICO: 'nao-ironico', IRONICO: 'ironico'}))

In [ ]:
# 5. Tokenizacao e divisao treino/teste.
from datasets import Dataset
from transformers import AutoTokenizer

MODELO_BASE = 'neuralmind/bert-base-portuguese-cased'
TAMANHO_MAXIMO = 192  # o mesmo de fraus/sinais/ironia.py

tokenizador = AutoTokenizer.from_pretrained(MODELO_BASE)

conjunto = Dataset.from_pandas(dados[['texto', 'rotulo']].reset_index(drop=True))
conjunto = conjunto.map(
    lambda lote: tokenizador(
        [str(t) for t in lote['texto']], truncation=True, max_length=TAMANHO_MAXIMO
    ),
    batched=True,
)
conjunto = conjunto.rename_column('rotulo', 'labels')
conjunto = conjunto.class_encode_column('labels')
divisao = conjunto.train_test_split(test_size=0.15, seed=42, stratify_by_column='labels')
print(divisao)

In [ ]:
# 6. Treino com pesos de classe.
import numpy as np
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, f1_score
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments)

modelo = AutoModelForSequenceClassification.from_pretrained(MODELO_BASE, num_labels=2)

# Corpus de ironia costuma ter muito mais exemplo nao-ironico. O peso evita que
# o modelo aprenda a responder "nao e ironia" sempre e ainda marcar boa acuracia.
contagem = np.bincount(divisao['train']['labels'], minlength=2)
pesos = len(divisao['train']) / (2 * np.maximum(contagem, 1))
pesos_tensor = torch.tensor(pesos, dtype=torch.float32).to('cuda')
print('contagem:', contagem, ' pesos:', pesos.round(3))


class TreinadorPonderado(Trainer):
    """Trainer com perda ponderada pelas duas classes."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        rotulos = inputs.pop('labels')
        saidas = model(**inputs)
        perda = nn.CrossEntropyLoss(weight=pesos_tensor)(
            saidas.logits.view(-1, 2), rotulos.view(-1)
        )
        return (perda, saidas) if return_outputs else perda


def calcular_metricas(predicao):
    preditos = np.argmax(predicao.predictions, axis=1)
    return {
        'acuracia': accuracy_score(predicao.label_ids, preditos),
        'f1_macro': f1_score(predicao.label_ids, preditos, average='macro'),
        # F1 da classe ironica isolada: e ela que o sinal usa, e a macro
        # esconderia um desempenho ruim nela atras da classe majoritaria.
        'f1_ironico': f1_score(predicao.label_ids, preditos, pos_label=IRONICO,
                               average='binary', zero_division=0),
    }


argumentos = TrainingArguments(
    output_dir=DIR_CHECKPOINT,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1_ironico',  # a classe que o sinal consome
    fp16=True,
    logging_steps=100,
)

treinador = TreinadorPonderado(
    model=modelo,
    args=argumentos,
    train_dataset=divisao['train'],
    eval_dataset=divisao['test'],
    data_collator=DataCollatorWithPadding(tokenizador),
    compute_metrics=calcular_metricas,
)
treinador.train()

In [ ]:
# 7. Avaliacao e sanidade em frases de atendimento.
predicao = treinador.predict(divisao['test'])
preditos = np.argmax(predicao.predictions, axis=1)

print(classification_report(
    predicao.label_ids, preditos,
    labels=[NAO_IRONICO, IRONICO],
    target_names=['nao-ironico', 'ironico'],
    digits=4,
    zero_division=0,
))

# O IDPT e de tweets e noticias, NAO de atendimento. Estas frases nao sao
# metrica -- sao um cheiro de transferencia de dominio antes de exportar.
import torch.nn.functional as F

SONDAS = [
    ('otimo servico, so esperei 3 horas', 'esperado: ironico'),
    ('parabens pelo atendimento, nota mil, so 5 dias sem resposta', 'esperado: ironico'),
    ('qual o prazo de entrega do meu pedido', 'esperado: nao-ironico'),
    ('obrigado, resolveram rapido', 'esperado: nao-ironico'),
]
entradas = tokenizador([t for t, _ in SONDAS], return_tensors='pt',
                       truncation=True, max_length=TAMANHO_MAXIMO, padding=True).to('cuda')
modelo.eval()
with torch.inference_mode():
    probabilidades = F.softmax(modelo(**entradas).logits, dim=-1)[:, IRONICO].tolist()

print('sondas de dominio (nao sao metrica):')
for (frase, esperado), p in zip(SONDAS, probabilidades):
    print(f'  P(ironico)={p:.3f}  {frase!r}  ({esperado})')

In [ ]:
# 8. Export dos artefatos.
import json

metricas = treinador.evaluate()

modelo.save_pretrained(DIR_SAIDA)
tokenizador.save_pretrained(DIR_SAIDA)

# A PROCEDENCIA acompanha o artefato. Fixar 'IDPT 2021' aqui faria o
# metricas_ironia.json mentir sempre que ORIGEM fosse outra -- e a procedencia
# do corpus e justamente o que a banca vai perguntar.
PROCEDENCIA = {
    'sintetico': (
        f'sintetico: fraus.ingest.gerador_ironia, semente {SEMENTE} -- '
        'padroes declarados no modulo, NAO ironia humana observada'
    ),
    'drive': 'corpus colocado a mao em DIR_CORPUS (declare a fonte no relatorio)',
    'kaggle': f'Kaggle: {DATASET_KAGGLE} -- rotulo AUTO-ATRIBUIDO por hashtag',
    'upload': 'arquivo enviado pelo navegador (declare a fonte no relatorio)',
}
LIMITACAO = {
    'sintetico': (
        'corpus GERADO: a ironia segue padroes declarados em '
        'fraus/ingest/gerador_ironia.py. Ironia real e mais variada e '
        'depende de contexto que a frase isolada nao carrega -- a tese de '
        'Vieira e Silva (USP, 2025) mediu que ~30% dos exemplos exigiram o '
        'contexto de producao ate para o anotador HUMANO. Esta metrica e '
        'OTIMISTA por construcao'
    ),
    'drive': 'transferencia de dominio para atendimento de chatbot nao verificada',
    'kaggle': (
        'rotulo auto-atribuido pelo autor do tweet (hashtag), nao anotado por '
        'terceiro: ha ironia sem hashtag e hashtag sem ironia. Alem disso, '
        'tweets nao sao atendimento -- transferencia de dominio nao verificada'
    ),
    'upload': 'transferencia de dominio para atendimento de chatbot nao verificada',
}

with open(f'{DIR_MODELOS}/metricas_ironia.json', 'w', encoding='utf-8') as arquivo:
    json.dump({
        'acuracia': float(metricas['eval_acuracia']),
        'f1_macro': float(metricas['eval_f1_macro']),
        'f1_ironico': float(metricas['eval_f1_ironico']),
        'classes': ['nao-ironico', 'ironico'],
        'corpus': PROCEDENCIA[ORIGEM],
        'exemplos_treino': int(len(divisao['train'])),
        'limitacao': LIMITACAO[ORIGEM],
    }, arquivo, ensure_ascii=False, indent=2)

print('gravados:')
print(' ', DIR_SAIDA)
print(' ', f'{DIR_MODELOS}/metricas_ironia.json')

# Sem Drive, o que esta em /content morre com a sessao: baixa agora.
if not RAIZ.startswith('/content/drive'):
    print('\nSem Drive -- compactando para download imediato...')
    shutil.make_archive('/content/bertimbau-ironia', 'zip', DIR_SAIDA)
    from google.colab import files

    files.download('/content/bertimbau-ironia.zip')
    files.download(f'{DIR_MODELOS}/metricas_ironia.json')
else:
    print()
    print('Copie para o repo local:')
    print(f'  {DIR_SAIDA}  ->  modelos/bertimbau-ironia/')
    print(f'  {DIR_MODELOS}/metricas_ironia.json  ->  modelos/metricas_ironia.json')
